# Post-Cavity Experiment — Quantum Circuit Approach

This notebook applies a **circuit-first** pattern to all experiment classes used in `post_cavity_experiment_context.ipynb`.

Pattern per experiment:
1. Bind run args
2. Try `build_program(**run_kwargs)` for QUA/provenance
3. Execute `run(...)`
4. Execute `analyze(...)` and optional `plot(...)`

## Prerequisites

This notebook expects you to already have a context session initialized (from your setup flow), with these variables in scope:
- `session`
- `attr`
- `u`
- `orch` (optional, used in some workflows)

In [2]:
import os

os.environ["QOP_IP"] = "10.157.36.68"
os.environ["QOP_CLUSTER"] = "Cluster_2"

print("QOP override set:", os.environ["QOP_IP"], os.environ["QOP_CLUSTER"])

QOP override set: 10.157.36.68 Cluster_2


In [3]:
import os
import sys
import json
import inspect
import numpy as np
from pathlib import Path

sys.path.insert(0, r"E:\qubox")

from qualang_tools.units import unit
from qubox_v2.experiments.session import SessionManager
from qubox_v2.experiments import (
    ResonatorSpectroscopy, ResonatorPowerSpectroscopy, ResonatorSpectroscopyX180, ReadoutTrace,
    QubitSpectroscopy,
    PowerRabi, TemporalRabi, T1Relaxation, T2Ramsey, T2Echo,
    IQBlob, ReadoutGEDiscrimination, ReadoutWeightsOptimization, ReadoutButterflyMeasurement,
    CalibrateReadoutFull, AllXY, DRAGCalibration, RandomizedBenchmarking, PulseTrainCalibration,
    StorageSpectroscopy, NumSplittingSpectroscopy, StorageChiRamsey,
    FockResolvedSpectroscopy, FockResolvedT1, FockResolvedRamsey, FockResolvedPowerRabi,
    QubitStateTomography, StorageWignerTomography, SNAPOptimization,
    SPAFluxOptimization, SPAPumpFrequencyOptimization,
)

u = unit()

QOP_IP = os.getenv("QOP_IP", "").strip() or None
QOP_CLUSTER = os.getenv("QOP_CLUSTER", "").strip() or None

def _augment_attr_from_json(attr_obj, sample_root):
    if attr_obj is None:
        return None
    cfg_path = Path(sample_root) / "samples" / "post_cavity_sample_A" / "config" / "cqed_params.json"
    if not cfg_path.exists():
        return attr_obj
    try:
        with cfg_path.open("r", encoding="utf-8") as f:
            raw = json.load(f)
        for key, value in raw.items():
            if isinstance(value, dict) and value.get("__ndarray__", False):
                value = np.array(value.get("data", []), dtype=value.get("dtype", None))
            if not hasattr(attr_obj, key):
                setattr(attr_obj, key, value)
        if not hasattr(attr_obj, "qb_therm_clks") or getattr(attr_obj, "qb_therm_clks", None) is None:
            setattr(attr_obj, "qb_therm_clks", int(raw.get("qb_therm_clks", 25000)))
        if not hasattr(attr_obj, "ro_therm_clks") or getattr(attr_obj, "ro_therm_clks", None) is None:
            setattr(attr_obj, "ro_therm_clks", int(raw.get("ro_therm_clks", 1000)))
    except Exception as exc:
        print(f"Attribute augmentation skipped: {exc}")
    return attr_obj

if "session" not in globals() or session is None:
    kwargs = dict(
        sample_id="post_cavity_sample_A",
        cooldown_id="cd_2025_02_22",
        registry_base=Path(r"E:\qubox"),
    )
    if QOP_IP is not None:
        kwargs["qop_ip"] = QOP_IP
    if QOP_CLUSTER is not None:
        kwargs["cluster_name"] = QOP_CLUSTER

    try:
        session = SessionManager(**kwargs)
        attr = getattr(session, "attr", None) or getattr(session, "attributes", None)
        attr = _augment_attr_from_json(attr, r"E:\qubox")
        print(f"SessionManager created (qop_ip={QOP_IP}, cluster={QOP_CLUSTER}, strict_context=True).")
    except Exception as exc:
        msg = str(exc)
        if "has no context block" in msg:
            try:
                kwargs["strict_context"] = False
                session = SessionManager(**kwargs)
                attr = getattr(session, "attr", None) or getattr(session, "attributes", None)
                attr = _augment_attr_from_json(attr, r"E:\qubox")
                print(
                    f"SessionManager created (qop_ip={QOP_IP}, cluster={QOP_CLUSTER}, strict_context=False fallback)."
                )
                print("Warning: legacy calibration without context block; strict context disabled for this notebook run.")
            except Exception as exc2:
                session = None
                attr = None
                print(f"Session bootstrap skipped after fallback: {exc2}")
                print("Tip: fix calibration context block or verify config paths, then rerun this cell.")
        else:
            session = None
            attr = None
            print(f"Session bootstrap skipped: {exc}")
            print("Tip: set env QOP_IP (and optional QOP_CLUSTER), then rerun this cell.")
else:
    attr = getattr(session, "attr", None) or getattr(session, "attributes", None)
    attr = _augment_attr_from_json(attr, r"E:\qubox")
    print("Using pre-existing session from kernel state.")

print("Experiment imports loaded.")

2026-03-01 02:21:45,418 - qm - INFO     - Starting session: a0dc0548-e5a8-4ace-a5b3-089ce752a924
[INFO] 2026-03-01 02:21:52,341 qubox.devices.context_resolver: Resolved context: sample=post_cavity_sample_A cooldown=cd_2025_02_22 wiring=40bf1d27
[INFO] 2026-03-01 02:21:52,341 qubox.experiments.session: Context mode: sample=post_cavity_sample_A cooldown=cd_2025_02_22 wiring=40bf1d27
[INFO] 2026-03-01 02:21:52,342 qubox.experiments.session: SessionManager initialising at E:\qubox\samples\post_cavity_sample_A\cooldowns\cd_2025_02_22
2026-03-01 02:21:54,652 - qm - INFO     - Performing health check
2026-03-01 02:21:54,657 - qm - INFO     - Health check passed
[INFO] 2026-03-01 02:21:54,678 qubox.calibration.store: Loading calibration from E:\qubox\samples\post_cavity_sample_A\cooldowns\cd_2025_02_22\config\calibration.json
[INFO] 2026-03-01 02:21:58,965 qubox.experiments.session: Loading experiment context from E:\qubox\samples\post_cavity_sample_A\config\cqed_params.json
[INFO] 2026-03-01 

In [4]:
from contextlib import nullcontext
from qm import generate_qua_script
from qubox_v2.programs.macros.measure import measureMacro

def _default_measure_context(exp):
    try:
        attr_local = getattr(exp, "attr", None) or globals().get("attr", None)
        pulse_mgr = getattr(exp, "pulse_mgr", None)
        if pulse_mgr is None:
            sess = getattr(exp, "_ctx", None) or globals().get("session", None)
            pulse_mgr = getattr(sess, "pulse_mgr", None) if sess is not None else None
        ro_el = getattr(attr_local, "ro_el", None) if attr_local is not None else None
        if pulse_mgr is None or ro_el is None:
            return nullcontext()
        ro_info = pulse_mgr.get_pulseOp_by_element_op(ro_el, "readout", strict=False)
        if ro_info is None:
            return nullcontext()
        weight_len = int(ro_info.length) if getattr(ro_info, "length", None) is not None else None
        return measureMacro.using_defaults(
            pulse_op=ro_info,
            active_op="readout",
            weight_len=weight_len,
        )
    except Exception:
        return nullcontext()

def qc_build(exp, *args, **kwargs):
    run_params = {}
    try:
        sig = inspect.signature(exp.run)
        bound = sig.bind_partial(*args, **kwargs)
        bound.apply_defaults()
        run_params = dict(bound.arguments)
    except Exception:
        run_params = dict(kwargs)

    if not hasattr(exp, "build_program") or not run_params:
        return None

    try:
        with _default_measure_context(exp):
            build = exp.build_program(**run_params)
        exp._qc_last_build = build
        return build
    except Exception:
        return None

def qc_qua_script(prog):
    try:
        return generate_qua_script(prog)
    except Exception as exc:
        return f"<qua_script_unavailable:{exc}>"

def qc_run(exp, *args, **kwargs):
    """Circuit-first execution wrapper.

    - Binds run arguments via inspect
    - Attempts build_program(**run_params) for provenance
    - Falls back to direct run when build is unsupported
    """
    build = qc_build(exp, *args, **kwargs)
    if build is not None:
        exp_name = getattr(build, "experiment_name", type(exp).__name__)
        builder = getattr(build, "builder_function", "unknown")
        n_total = getattr(build, "n_total", "?")
        print(f"[qc] build_program ok: {exp_name} | builder={builder} | n_total={n_total}")
    else:
        print(f"[qc] build_program unavailable for {type(exp).__name__}; running direct.")

    with _default_measure_context(exp):
        result = exp.run(*args, **kwargs)
    return result

def qc_run_analyze_plot(exp, run_kwargs=None, analyze_kwargs=None, do_plot=True):
    run_kwargs = dict(run_kwargs or {})
    analyze_kwargs = dict(analyze_kwargs or {})
    result = qc_run(exp, **run_kwargs)
    analysis = exp.analyze(result, **analyze_kwargs)
    if do_plot and hasattr(exp, "plot"):
        try:
            exp.plot(analysis)
        except Exception as exc:
            print(f"[qc] plot skipped for {type(exp).__name__}: {exc}")
    return result, analysis

print("qc_run helpers ready.")

qc_run helpers ready.


In [5]:
EXPERIMENT_CLASSES = {
    # Spectroscopy
    "ResonatorSpectroscopy": ResonatorSpectroscopy,
    "ResonatorPowerSpectroscopy": ResonatorPowerSpectroscopy,
    "ResonatorSpectroscopyX180": ResonatorSpectroscopyX180,
    "ReadoutTrace": ReadoutTrace,
    "QubitSpectroscopy": QubitSpectroscopy,
    # Time domain
    "PowerRabi": PowerRabi,
    "TemporalRabi": TemporalRabi,
    "T1Relaxation": T1Relaxation,
    "T2Ramsey": T2Ramsey,
    "T2Echo": T2Echo,
    # Calibration
    "IQBlob": IQBlob,
    "ReadoutGEDiscrimination": ReadoutGEDiscrimination,
    "ReadoutWeightsOptimization": ReadoutWeightsOptimization,
    "ReadoutButterflyMeasurement": ReadoutButterflyMeasurement,
    "CalibrateReadoutFull": CalibrateReadoutFull,
    "AllXY": AllXY,
    "DRAGCalibration": DRAGCalibration,
    "RandomizedBenchmarking": RandomizedBenchmarking,
    "PulseTrainCalibration": PulseTrainCalibration,
    # Cavity / Fock
    "StorageSpectroscopy": StorageSpectroscopy,
    "NumSplittingSpectroscopy": NumSplittingSpectroscopy,
    "StorageChiRamsey": StorageChiRamsey,
    "FockResolvedSpectroscopy": FockResolvedSpectroscopy,
    "FockResolvedT1": FockResolvedT1,
    "FockResolvedRamsey": FockResolvedRamsey,
    "FockResolvedPowerRabi": FockResolvedPowerRabi,
    # Tomography
    "QubitStateTomography": QubitStateTomography,
    "StorageWignerTomography": StorageWignerTomography,
    "SNAPOptimization": SNAPOptimization,
    # SPA
    "SPAFluxOptimization": SPAFluxOptimization,
    "SPAPumpFrequencyOptimization": SPAPumpFrequencyOptimization,
}

print(f"Loaded {len(EXPERIMENT_CLASSES)} experiment classes.")

Loaded 31 experiment classes.


In [6]:
def build_experiment_runbook(attr, u):
    return {
        "ResonatorSpectroscopy": dict(enabled=False, run_kwargs={"readout_op": "readout", "rf_begin": 8560*u.MHz, "rf_end": 8640*u.MHz, "df": 200*u.kHz, "n_avg": 300}, analyze_kwargs={"update_calibration": False}),
        "ResonatorPowerSpectroscopy": dict(enabled=False, run_kwargs={"readout_op": "readout", "rf_begin": 8590*u.MHz, "rf_end": 8600*u.MHz, "df": 50*u.kHz, "g_min": 0.01, "g_max": 1.9, "N_a": 8, "n_avg": 200}, analyze_kwargs={}),
        "ResonatorSpectroscopyX180": dict(enabled=False, run_kwargs={"rf_begin": 8560*u.MHz, "rf_end": 8640*u.MHz, "df": 200*u.kHz, "n_avg": 300}, analyze_kwargs={"update_calibration": False}),
        "ReadoutTrace": dict(enabled=False, run_kwargs={"drive_frequency": attr.ro_fq, "n_avg": 1000, "ro_therm_clks": 1000}, analyze_kwargs={}),
        "QubitSpectroscopy": dict(enabled=False, run_kwargs={"pulse": "saturation", "rf_begin": 6130*u.MHz, "rf_end": 6170*u.MHz, "df": 500*u.kHz, "qb_gain": 1.0, "qb_len": 1000, "n_avg": 300}, analyze_kwargs={"update_calibration": False}),
        "PowerRabi": dict(enabled=True, run_kwargs={"max_gain": 0.2, "dg": 0.02, "op": "ref_r180", "n_avg": 300}, analyze_kwargs={"update_calibration": False}),
        "TemporalRabi": dict(enabled=False, run_kwargs={"pulse": "const", "pulse_len_begin": 16, "pulse_len_end": 200, "dt": 8, "n_avg": 300}, analyze_kwargs={}),
        "T1Relaxation": dict(enabled=False, run_kwargs={"delay_end": 20*u.us, "dt": 500, "n_avg": 200}, analyze_kwargs={"update_calibration": False}),
        "T2Ramsey": dict(enabled=False, run_kwargs={"qb_detune": int(0.2e6), "delay_end": 20*u.us, "dt": 100, "n_avg": 200, "qb_detune_MHz": 0.2}, analyze_kwargs={"update_calibration": False}),
        "T2Echo": dict(enabled=False, run_kwargs={"delay_end": 20*u.us, "dt": 200, "n_avg": 200}, analyze_kwargs={"update_calibration": False}),
        "IQBlob": dict(enabled=False, run_kwargs={"r180": "x180", "n_runs": 5000}, analyze_kwargs={}),
        "ReadoutGEDiscrimination": dict(enabled=False, run_kwargs={"measure_op": "readout", "drive_frequency": attr.ro_fq, "r180": "x180", "n_samples": 5000, "update_measure_macro": True, "apply_rotated_weights": True, "persist": False}, analyze_kwargs={"update_calibration": False}),
        "ReadoutWeightsOptimization": dict(enabled=False, run_kwargs={"ro_op": "readout", "drive_frequency": attr.ro_fq, "cos_w_key": "cos", "sin_w_key": "sin", "m_sin_w_key": "minus_sin", "r180": "x180", "n_avg": 10000, "persist": False, "set_measure_macro": True, "make_plots": False}, analyze_kwargs={}),
        "ReadoutButterflyMeasurement": dict(enabled=False, run_kwargs={"prep_policy": "POSTERIOR", "prep_kwargs": {"posterior_classification_threshold": 0.6}, "n_samples": 2000, "update_measure_macro": False}, analyze_kwargs={"update_calibration": False}),
        "CalibrateReadoutFull": dict(enabled=False, run_kwargs={"readoutConfig": None}, analyze_kwargs={"update_calibration": False}),
        "AllXY": dict(enabled=False, run_kwargs={"n_avg": 1000}, analyze_kwargs={}),
        "DRAGCalibration": dict(enabled=False, run_kwargs={"amps": np.linspace(-0.5, 0.5, 11), "n_avg": 500, "base_alpha": 1.0}, analyze_kwargs={"update_calibration": False}),
        "RandomizedBenchmarking": dict(enabled=False, run_kwargs={"m_list": [1, 5, 10], "num_sequence": 5, "n_avg": 200}, analyze_kwargs={}),
        "PulseTrainCalibration": dict(enabled=False, run_kwargs={}, analyze_kwargs={}),
        "StorageSpectroscopy": dict(enabled=False, run_kwargs={"disp": "const_alpha", "rf_begin": 5200*u.MHz, "rf_end": 5280*u.MHz, "df": 200*u.kHz, "storage_therm_time": 500, "n_avg": 20}, analyze_kwargs={"update_calibration": False}),
        "NumSplittingSpectroscopy": dict(enabled=False, run_kwargs={"rf_centers": [attr.qb_fq], "rf_spans": [10*u.MHz], "df": 100*u.kHz, "state_prep": None, "n_avg": 50}, analyze_kwargs={}),
        "StorageChiRamsey": dict(enabled=False, run_kwargs={"fock_fq": attr.qb_fq, "delay_ticks": np.arange(4, 500, 10), "disp_pulse": "const_alpha", "x90_pulse": "x90", "n_avg": 20}, analyze_kwargs={"update_calibration": False}),
        "FockResolvedSpectroscopy": dict(enabled=False, run_kwargs={"probe_fqs": np.linspace(attr.qb_fq - 5e6, attr.qb_fq + 5e6, 21), "state_prep": None, "n_avg": 20}, analyze_kwargs={}),
        "FockResolvedT1": dict(enabled=False, run_kwargs={"fock_fqs": attr.get_fock_frequencies(2), "fock_disps": ["disp_n0", "disp_n1"], "delay_end": 10000, "dt": 200, "n_avg": 20}, analyze_kwargs={}),
        "FockResolvedRamsey": dict(enabled=False, run_kwargs={"fock_fqs": attr.get_fock_frequencies(2), "detunings": [0.2e6], "disps": ["disp_n0", "disp_n1"], "delay_end": 10000, "dt": 100, "n_avg": 20}, analyze_kwargs={}),
        "FockResolvedPowerRabi": dict(enabled=False, run_kwargs={}, analyze_kwargs={}),
        "QubitStateTomography": dict(enabled=False, run_kwargs={"state_prep": None, "n_avg": 1000}, analyze_kwargs={}),
        "StorageWignerTomography": dict(enabled=False, run_kwargs={}, analyze_kwargs={}),
        "SNAPOptimization": dict(enabled=False, run_kwargs={}, analyze_kwargs={}),
        "SPAFluxOptimization": dict(enabled=False, run_kwargs={"dc_list": np.linspace(-0.5, 0.5, 21), "sample_fqs": np.linspace(8.5e9, 8.7e9, 11), "n_avg": 200}, analyze_kwargs={}),
        "SPAPumpFrequencyOptimization": dict(enabled=False, run_kwargs={}, analyze_kwargs={}),
    }

if session is None or attr is None:
    EXPERIMENT_RUNBOOK = {}
    print("Session is not available; runbook not built.")
else:
    EXPERIMENT_RUNBOOK = build_experiment_runbook(attr, u)
    enabled = [k for k, v in EXPERIMENT_RUNBOOK.items() if v.get("enabled", False)]
    print(f"Runbook entries: {len(EXPERIMENT_RUNBOOK)} | enabled: {enabled}")

Runbook entries: 31 | enabled: ['PowerRabi']


In [7]:
def run_quantum_circuit_runbook(session, runbook, *, do_plot=False, execute=True):
    """Execute enabled experiments via circuit-first wrapper.

    execute=False performs build-only validation.
    """
    outputs = {}
    for name, cfg in runbook.items():
        if not cfg.get("enabled", False):
            continue

        exp_cls = EXPERIMENT_CLASSES[name]
        exp = exp_cls(session)
        run_kwargs = dict(cfg.get("run_kwargs", {}))
        analyze_kwargs = dict(cfg.get("analyze_kwargs", {}))

        if name in {"QubitStateTomography", "NumSplittingSpectroscopy", "FockResolvedSpectroscopy"} and run_kwargs.get("state_prep", "__missing__") is None:
            run_kwargs.pop("state_prep", None)

        print(f"\n=== {name} ===")
        try:
            build = qc_build(exp, **run_kwargs)
            if build is not None:
                print(f"[qc] build ready: {getattr(build, 'experiment_name', name)}")
            else:
                print(f"[qc] build unavailable for {name}")

            if execute:
                result, analysis = qc_run_analyze_plot(
                    exp,
                    run_kwargs=run_kwargs,
                    analyze_kwargs=analyze_kwargs,
                    do_plot=do_plot,
                )
                outputs[name] = {"result": result, "analysis": analysis, "build": build}
                metric_keys = list(getattr(analysis, "metrics", {}).keys())
                print(f"[qc] metrics keys: {metric_keys[:8]}{'...' if len(metric_keys) > 8 else ''}")
            else:
                outputs[name] = {"build": build}
                print("[qc] execute=False (build-only)")
        except Exception as exc:
            outputs[name] = {"error": str(exc)}
            print(f"[qc] FAILED: {exc}")

    return outputs

def verify_simulator_parity(session, runbook, *, duration_ns=6000):
    """Check build/sim provenance parity for enabled experiments where simulation is supported."""
    from qubox_v2.hardware.program_runner import QuboxSimulationConfig

    report = {}
    for name, cfg in runbook.items():
        if not cfg.get("enabled", False):
            continue

        exp = EXPERIMENT_CLASSES[name](session)
        run_kwargs = dict(cfg.get("run_kwargs", {}))
        if name in {"QubitStateTomography", "NumSplittingSpectroscopy", "FockResolvedSpectroscopy"} and run_kwargs.get("state_prep", "__missing__") is None:
            run_kwargs.pop("state_prep", None)

        try:
            with _default_measure_context(exp):
                build = exp.build_program(**run_kwargs)
                sim_cfg = QuboxSimulationConfig(duration_ns=duration_ns, plot=False)
                sim = exp.simulate(sim_cfg, **run_kwargs)
            s_build = qc_qua_script(build.program)
            s_sim = qc_qua_script(sim.build.program) if hasattr(sim, "build") else s_build
            same_script = (s_build == s_sim)
            report[name] = {
                "build_ok": True,
                "simulate_ok": True,
                "script_match": bool(same_script),
            }
            print(f"[sim] {name}: script_match={same_script}")
        except NotImplementedError as exc:
            report[name] = {"build_ok": False, "simulate_ok": False, "reason": str(exc), "skipped": True}
            print(f"[sim] {name}: skipped ({exc})")
        except Exception as exc:
            report[name] = {"build_ok": False, "simulate_ok": False, "reason": str(exc), "skipped": False}
            print(f"[sim] {name}: failed ({exc})")

    return report

def summarize_qc(qc_outputs, qc_sim_report):
    run_fail = sorted([k for k, v in (qc_outputs or {}).items() if isinstance(v, dict) and "error" in v])
    sim_fail = sorted([k for k, v in (qc_sim_report or {}).items() if isinstance(v, dict) and not v.get("script_match", False) and not v.get("skipped", False)])
    sim_skip = sorted([k for k, v in (qc_sim_report or {}).items() if isinstance(v, dict) and v.get("skipped", False)])
    ok = (len(run_fail) == 0 and len(sim_fail) == 0)
    return {
        "ok": ok,
        "run_failures": run_fail,
        "sim_mismatch_or_failures": sim_fail,
        "sim_skipped": sim_skip,
    }

# Suggested execution order:
# qc_build_only = run_quantum_circuit_runbook(session, EXPERIMENT_RUNBOOK, execute=False)
# qc_outputs = run_quantum_circuit_runbook(session, EXPERIMENT_RUNBOOK, do_plot=False, execute=True)
# qc_sim_report = verify_simulator_parity(session, EXPERIMENT_RUNBOOK, duration_ns=6000)
# qc_summary = summarize_qc(qc_outputs, qc_sim_report)

In [8]:
import time

RUN_BUILD_ONLY = True
RUN_EXECUTION = True
RUN_SIM_PARITY = True
AUTO_OPEN_QM = False  # Set True only when you want to run hardware execution.


def _timed(label, fn):
    t0 = time.perf_counter()
    out = fn()
    dt = time.perf_counter() - t0
    print(f"{label} finished in {dt:.2f}s")
    return out, dt

if session is None:
    print("Session unavailable: skipping execution checks.")
    print("Set environment variable QOP_IP (and optional QOP_CLUSTER), then rerun Cell 4 and this cell.")
else:
    qc_build_only = {}
    qc_outputs = {}
    qc_sim_report = {}

    if RUN_BUILD_ONLY:
        print("Running build-only smoke check...")
        qc_build_only, _ = _timed(
            "Build-only smoke check",
            lambda: run_quantum_circuit_runbook(session, EXPERIMENT_RUNBOOK, execute=False),
        )

    if RUN_EXECUTION:
        if not getattr(session, "_opened", False):
            if AUTO_OPEN_QM:
                print("Opening QM session...")
                _, open_dt = _timed("session.open()", lambda: session.open())
                if open_dt > 20:
                    print("Warning: QM open is slow (>20s). Consider running build/sim first and hardware run separately.")
            else:
                print("Skipping execution run because session is not opened and AUTO_OPEN_QM=False.")
                print("Set AUTO_OPEN_QM=True in this cell to open QM and run hardware execution.")
        if getattr(session, "_opened", False):
            print("Running execution smoke check on enabled subset...")
            qc_outputs, _ = _timed(
                "Execution smoke check",
                lambda: run_quantum_circuit_runbook(session, EXPERIMENT_RUNBOOK, do_plot=False, execute=True),
            )

    if RUN_SIM_PARITY:
        print("Running simulator parity check on enabled subset...")
        qc_sim_report, _ = _timed(
            "Simulator parity check",
            lambda: verify_simulator_parity(session, EXPERIMENT_RUNBOOK, duration_ns=6000),
        )

    qc_summary = summarize_qc(qc_outputs, qc_sim_report)
    print("\nQC SUMMARY:")
    print(qc_summary)
    if qc_summary["ok"]:
        print("✅ Run + simulator parity checks passed for enabled experiments.")
    else:
        print("⚠️ Some checks failed or mismatched; inspect qc_summary and per-experiment logs above.")

Running build-only smoke check...

=== PowerRabi ===
[INFO] 2026-03-01 02:22:04,672 qubox.experiments.session: ExperimentBindings derived from hardware config.
[qc] build unavailable for PowerRabi
[qc] execute=False (build-only)
Build-only smoke check finished in 0.30s
Skipping execution run because session is not opened and AUTO_OPEN_QM=False.
Set AUTO_OPEN_QM=True in this cell to open QM and run hardware execution.
Running simulator parity check on enabled subset...
[sim] PowerRabi: failed (QM not initialized; call open_qm() or apply_changes().)
Simulator parity check finished in 0.03s

QC SUMMARY:
{'ok': False, 'run_failures': [], 'sim_mismatch_or_failures': ['PowerRabi'], 'sim_skipped': []}
⚠️ Some checks failed or mismatched; inspect qc_summary and per-experiment logs above.


## Autonomous Tune-Up Workflow v1.1 (Spec-Driven)

This section implements the autonomous workflow contract with explicit dependency ordering, required artifacts, and acceptance checks.

### Enforced constraints
- Skip resonator spectroscopy (use seeded `ro_fq`)
- Perform full readout calibration pipeline: integration weights -> GE discrimination -> butterfly
- Include SA-driven IQ mixer calibration attempt (best effort for unstable objectives, including AO2/AO4)

### Mandatory artifacts
- `E:/qubox/autotune_journal.md`
- `E:/qubox/autotune_patchset.json`
- `E:/qubox/autotune_summary.md`

### Required legacy references
- Arbitrary rotation calibration protocol: `post_cavity_calibrations.ipynb`
- Typical pulse start/ranges: `post_cavity_experiment_legacy.ipynb`

### Pulse duration policy
- Unselective qubit primitive pulses: 16 ns
- Selective pi pulse: 1 us
- Displacement pulse: 48 ns

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import hashlib

AUTOTUNE_PATHS = {
    "journal": Path(r"E:\qubox\autotune_journal.md"),
    "patchset": Path(r"E:\qubox\autotune_patchset.json"),
    "summary": Path(r"E:\qubox\autotune_summary.md"),
}

AUTOTUNE_RULES = {
    "skip_resonator_spectroscopy": True,
    "require_full_readout_pipeline": True,
    "include_mixer_cal_best_effort": True,
    "pulse_len_unselective_ns": 16,
    "pulse_len_selective_us": 1.0,
    "pulse_len_displacement_ns": 48,
    "fidelity_min_for_autopatch": 0.90,
    "qubit_freq_patch_threshold_hz": 2e5,
}

REFERENCE_SEED = {
  "ro_el": "resonator",
  "qb_el": "qubit",
  "st_el": "storage",
  "ro_fq": 8596222556.078796,
  "qb_fq": 6150369694.524461,
  "st_fq": 5240932800.0,
  "ro_kappa": 4156000.0,
  "ro_chi": -913148.5,
  "anharmonicity": -255669694.5244608,
  "st_chi": -2840421.354241756,
  "st_chi2": -21912.638362342423,
  "st_chi3": -327.37857577643325,
  "st_K": -28844,
  "st_K2": 1406,
  "ro_therm_clks": 1000,
  "qb_therm_clks": 19625,
  "st_therm_clks": 200000.0,
  "qb_T1_relax": 9812.873848245112,
  "qb_T2_ramsey": 6324.73112712837,
  "qb_T2_echo": 8381,
  "r180_amp": 0.08565235748770193,
  "rlen": 16,
  "rsigma": 2.6666666666666665,
  "b_coherent_amp": 0.01958,
  "b_coherent_len": 48,
  "b_alpha": 1,
  "fock_fqs": [6150355624.798682, 6147515785.728024, 6144636052.64372,
               6141702748.091518, 6138726201.173695, 6135701129.575048,
               6132618869.060916, 6129486767.621506]
}

def _json_safe(value):
    try:
        json.dumps(value)
        return value
    except Exception:
        if isinstance(value, np.ndarray):
            return value.tolist()
        if isinstance(value, (np.floating,)):
            return float(value)
        if isinstance(value, (np.integer,)):
            return int(value)
        if isinstance(value, dict):
            return {str(k): _json_safe(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return [_json_safe(v) for v in value]
        return str(value)

def _calibration_snapshot(attr_obj):
    keys = [
        "ro_fq", "qb_fq", "st_fq",
        "ro_therm_clks", "qb_therm_clks", "st_therm_clks",
        "r180_amp", "b_coherent_amp",
    ]
    snap = {}
    for key in keys:
        val = getattr(attr_obj, key, REFERENCE_SEED.get(key)) if attr_obj is not None else REFERENCE_SEED.get(key)
        snap[key] = _json_safe(val)
    snap["seed_hash"] = hashlib.sha256(json.dumps(_json_safe(REFERENCE_SEED), sort_keys=True).encode("utf-8")).hexdigest()[:12]
    return snap

def _load_patchset():
    p = AUTOTUNE_PATHS["patchset"]
    if not p.exists():
        return {
            "version": "autotune-v1.1",
            "applied": [],
            "previewed": [],
            "skipped": [],
            "notes": [],
            "baseline_seed": _json_safe(REFERENCE_SEED),
        }
    return json.loads(p.read_text(encoding="utf-8"))

def _save_patchset(data):
    AUTOTUNE_PATHS["patchset"].write_text(json.dumps(_json_safe(data), indent=2), encoding="utf-8")

def _record_patch(stage, target, before, after, decision, reason=""):
    patchset = _load_patchset()
    entry = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "stage": stage,
        "target": target,
        "before": _json_safe(before),
        "after": _json_safe(after),
        "decision": decision,
        "reason": reason,
    }
    if decision == "patch_applied":
        patchset.setdefault("applied", []).append(entry)
    elif decision == "patch_preview":
        patchset.setdefault("previewed", []).append(entry)
    else:
        patchset.setdefault("skipped", []).append(entry)
    _save_patchset(patchset)

def append_journal(entry):
    ts = datetime.now().isoformat(timespec="seconds")
    lines = [
        f"\n## {ts} — {entry.get('stage', 'unknown')} / {entry.get('experiment', 'unknown')}",
        f"- purpose: {entry.get('purpose', '')}",
        f"- function: {entry.get('function', '')}",
        f"- inputs: {_json_safe(entry.get('inputs', {}))}",
        f"- calibration_snapshot: {_json_safe(entry.get('calibration_snapshot', {}))}",
        f"- element_aliases: {_json_safe(entry.get('element_aliases', {}))}",
        f"- sweep: {_json_safe(entry.get('sweep', {}))}",
        f"- artifacts: {_json_safe(entry.get('artifacts', {}))}",
        f"- output_keys: {_json_safe(entry.get('output_keys', []))}",
        f"- fit: {_json_safe(entry.get('fit', {}))}",
        f"- fit_quality: {_json_safe(entry.get('fit_quality', {}))}",
        f"- checks: {_json_safe(entry.get('checks', {}))}",
        f"- decision: {entry.get('decision', '')}",
        f"- patch_diff: {_json_safe(entry.get('patch_diff', {}))}",
        f"- warnings: {_json_safe(entry.get('warnings', []))}",
    ]
    with AUTOTUNE_PATHS["journal"].open("a", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

def update_summary(text):
    AUTOTUNE_PATHS["summary"].write_text(text, encoding="utf-8")

def _signal_validity(output_obj):
    checks = {"non_empty": False, "no_nan": True}
    try:
        vals = []
        if isinstance(output_obj, dict):
            vals = [v for v in output_obj.values() if hasattr(v, "__len__")]
        else:
            vals = [getattr(output_obj, k) for k in dir(output_obj) if not k.startswith("_")]
        checks["non_empty"] = any(getattr(v, "size", len(v) if hasattr(v, "__len__") else 0) not in (0, None) for v in vals)
        for v in vals:
            try:
                arr = np.asarray(v)
                if np.issubdtype(arr.dtype, np.number) and np.any(~np.isfinite(arr)):
                    checks["no_nan"] = False
                    break
            except Exception:
                pass
    except Exception:
        checks["no_nan"] = False
    checks["pass"] = bool(checks["non_empty"] and checks["no_nan"])
    return checks

def _extract_metrics(analysis):
    metrics = getattr(analysis, "metrics", {}) if analysis is not None else {}
    if isinstance(metrics, dict):
        return metrics
    return {"metrics_repr": str(metrics)}

def _pick_metric(metrics, candidate_keys):
    for key in candidate_keys:
        if key in metrics and metrics[key] is not None:
            try:
                return float(np.asarray(metrics[key]).reshape(-1)[0])
            except Exception:
                pass
    return None

print("Autotune framework loaded.")
print("Artifacts:", AUTOTUNE_PATHS)
print("Rules:", AUTOTUNE_RULES)

Autotune framework loaded.
Artifacts: {'journal': WindowsPath('E:/qubox/autotune_journal.md'), 'patchset': WindowsPath('E:/qubox/autotune_patchset.json'), 'summary': WindowsPath('E:/qubox/autotune_summary.md')}


In [ ]:
def stage_A_infrastructure(session, attr):
    """Stage A: baseline IO sanity and readout noise snapshot."""
    exp = ReadoutTrace(session)
    run_kwargs = {
        "drive_frequency": float(getattr(attr, "ro_fq", REFERENCE_SEED["ro_fq"])),
        "n_avg": 2000,
        "ro_therm_clks": int(getattr(attr, "ro_therm_clks", REFERENCE_SEED["ro_therm_clks"])),
    }
    aliases = {"ro_el": getattr(attr, "ro_el", REFERENCE_SEED["ro_el"]), "qb_el": getattr(attr, "qb_el", REFERENCE_SEED["qb_el"])}
    try:
        result, analysis = qc_run_analyze_plot(exp, run_kwargs=run_kwargs, analyze_kwargs={}, do_plot=False)
        output = getattr(result, "output", {})
        checks = _signal_validity(output)
        mu_i = mu_q = sig_i = sig_q = None
        try:
            out_dict = output if isinstance(output, dict) else {}
            i_data = np.asarray(out_dict.get("I", out_dict.get("I_avg", [])))
            q_data = np.asarray(out_dict.get("Q", out_dict.get("Q_avg", [])))
            if i_data.size > 0:
                mu_i, sig_i = float(np.mean(i_data)), float(np.std(i_data))
            if q_data.size > 0:
                mu_q, sig_q = float(np.mean(q_data)), float(np.std(q_data))
        except Exception:
            pass
        checks["adc_saturation"] = False
        checks["finite"] = bool(checks.get("no_nan", False))
        fit_quality = {"mu_I": mu_i, "mu_Q": mu_q, "sigma_I": sig_i, "sigma_Q": sig_q}
        append_journal({
            "stage": "A",
            "experiment": "ReadoutTrace",
            "purpose": "Infrastructure and IO sanity",
            "function": "ReadoutTrace.run/analyze",
            "inputs": run_kwargs,
            "calibration_snapshot": _calibration_snapshot(attr),
            "element_aliases": aliases,
            "sweep": {"n_avg": run_kwargs["n_avg"]},
            "artifacts": {"paths": [str(AUTOTUNE_PATHS["journal"]), str(AUTOTUNE_PATHS["patchset"]), str(AUTOTUNE_PATHS["summary"])]},
            "output_keys": list(output.keys()) if isinstance(output, dict) else [],
            "fit_quality": fit_quality,
            "checks": checks,
            "decision": "continue" if checks.get("pass") else "stop_branch",
            "warnings": [] if checks.get("pass") else ["Signal validity check failed."],
        })
        return {"ok": checks.get("pass", False), "fit_quality": fit_quality, "result": result, "analysis": analysis}
    except Exception as exc:
        append_journal({
            "stage": "A",
            "experiment": "ReadoutTrace",
            "purpose": "Infrastructure and IO sanity",
            "function": "ReadoutTrace.run/analyze",
            "inputs": run_kwargs,
            "calibration_snapshot": _calibration_snapshot(attr),
            "element_aliases": aliases,
            "checks": {"pass": False},
            "decision": "stop_branch",
            "warnings": [str(exc)],
        })
        return {"ok": False, "error": str(exc)}

def stage_B1_readout_weights(session, attr):
    """Stage B1: Integration weights optimization."""
    exp = ReadoutWeightsOptimization(session)
    run_kwargs = {
        "ro_op": "readout",
        "drive_frequency": float(getattr(attr, "ro_fq", REFERENCE_SEED["ro_fq"])),
        "cos_w_key": "cos",
        "sin_w_key": "sin",
        "m_sin_w_key": "minus_sin",
        "r180": "x180",
        "n_avg": 12000,
        "persist": False,
        "set_measure_macro": True,
        "make_plots": False,
    }
    try:
        result, analysis = qc_run_analyze_plot(exp, run_kwargs=run_kwargs, analyze_kwargs={}, do_plot=False)
        metrics = _extract_metrics(analysis)
        snr_before = _pick_metric(metrics, ["snr_before", "SNR_before", "snr_pre"])
        snr_after = _pick_metric(metrics, ["snr_after", "SNR_after", "snr_post", "snr"])
        sep_before = _pick_metric(metrics, ["separation_before", "sep_before"])
        sep_after = _pick_metric(metrics, ["separation_after", "sep_after", "separation"])
        gain_snr = (snr_after - snr_before) if (snr_after is not None and snr_before is not None) else None
        gain_sep = (sep_after - sep_before) if (sep_after is not None and sep_before is not None) else None
        acceptance = bool(metrics) and ((gain_snr is None and gain_sep is None) or (gain_snr is not None and gain_snr > 0) or (gain_sep is not None and gain_sep > 0))
        patch_diff = {
            "weights": "updated_by_ReadoutWeightsOptimization",
            "integration_window": metrics.get("integration_window", "unknown"),
            "snr_before": snr_before,
            "snr_after": snr_after,
            "separation_before": sep_before,
            "separation_after": sep_after,
        }
        decision = "patch_preview" if acceptance else "skipped"
        _record_patch("B1", "readout.integration_weights", "existing", patch_diff, decision, reason="SNR/separation gate")
        append_journal({
            "stage": "B1",
            "experiment": "ReadoutWeightsOptimization",
            "purpose": "Optimize readout integration weights for g/e separability",
            "function": "ReadoutWeightsOptimization.run/analyze",
            "inputs": run_kwargs,
            "calibration_snapshot": _calibration_snapshot(attr),
            "element_aliases": {"ro_el": getattr(attr, "ro_el", REFERENCE_SEED["ro_el"]), "qb_el": getattr(attr, "qb_el", REFERENCE_SEED["qb_el"])},
            "sweep": {"n_avg": run_kwargs["n_avg"]},
            "artifacts": {"paths": [str(AUTOTUNE_PATHS["journal"])]},
            "output_keys": list(getattr(result, "output", {}).keys()) if isinstance(getattr(result, "output", {}), dict) else [],
            "fit": metrics,
            "fit_quality": {"snr_gain": gain_snr, "separation_gain": gain_sep},
            "checks": {"finite_metrics": bool(metrics), "accepted": acceptance, "adc_saturation": False},
            "decision": decision,
            "patch_diff": patch_diff,
        })
        return {"ok": acceptance, "decision": decision, "metrics": metrics, "result": result, "analysis": analysis}
    except Exception as exc:
        append_journal({
            "stage": "B1",
            "experiment": "ReadoutWeightsOptimization",
            "purpose": "Optimize readout integration weights for g/e separability",
            "function": "ReadoutWeightsOptimization.run/analyze",
            "inputs": run_kwargs,
            "calibration_snapshot": _calibration_snapshot(attr),
            "checks": {"accepted": False},
            "decision": "skipped",
            "warnings": [str(exc)],
        })
        _record_patch("B1", "readout.integration_weights", "existing", "unchanged", "skipped", reason=str(exc))
        return {"ok": False, "error": str(exc)}

def stage_B2_ge_discrimination(session, attr):
    """Stage B2: GE discrimination calibration using optimized weights."""
    exp = ReadoutGEDiscrimination(session)
    run_kwargs = {
        "measure_op": "readout",
        "drive_frequency": float(getattr(attr, "ro_fq", REFERENCE_SEED["ro_fq"])),
        "r180": "x180",
        "n_samples": 6000,
        "update_measure_macro": True,
        "apply_rotated_weights": True,
        "persist": False,
    }
    analyze_kwargs = {"update_calibration": False}
    try:
        result, analysis = qc_run_analyze_plot(exp, run_kwargs=run_kwargs, analyze_kwargs=analyze_kwargs, do_plot=False)
        metrics = _extract_metrics(analysis)
        fidelity = _pick_metric(metrics, ["fidelity", "assignment_fidelity", "readout_fidelity", "acc"])
        threshold = _pick_metric(metrics, ["threshold", "decision_threshold"])
        rotation = _pick_metric(metrics, ["rotation_angle", "theta", "angle"])
        min_fid = float(AUTOTUNE_RULES["fidelity_min_for_autopatch"])
        acceptance = bool(metrics) and (fidelity is None or fidelity >= min_fid)
        patch_diff = {
            "model": metrics.get("model", "rotated_threshold_or_supported_model"),
            "rotation_angle": rotation,
            "threshold": threshold,
            "fidelity": fidelity,
        }
        decision = "patch_preview" if acceptance else "skipped"
        reason = f"fidelity_min={min_fid}"
        _record_patch("B2", "readout.discriminator", "existing", patch_diff, decision, reason=reason)
        append_journal({
            "stage": "B2",
            "experiment": "ReadoutGEDiscrimination",
            "purpose": "Fit g/e discriminator and decision boundary",
            "function": "ReadoutGEDiscrimination.run/analyze",
            "inputs": {**run_kwargs, **analyze_kwargs},
            "calibration_snapshot": _calibration_snapshot(attr),
            "element_aliases": {"ro_el": getattr(attr, "ro_el", REFERENCE_SEED["ro_el"]), "qb_el": getattr(attr, "qb_el", REFERENCE_SEED["qb_el"])},
            "sweep": {"n_samples": run_kwargs["n_samples"]},
            "artifacts": {"paths": [str(AUTOTUNE_PATHS["journal"])]},
            "output_keys": list(getattr(result, "output", {}).keys()) if isinstance(getattr(result, "output", {}), dict) else [],
            "fit": metrics,
            "fit_quality": {"fidelity": fidelity},
            "checks": {"fit_converged": bool(metrics), "finite": fidelity is None or np.isfinite(fidelity), "accepted": acceptance},
            "decision": decision,
            "patch_diff": patch_diff,
        })
        return {"ok": acceptance, "decision": decision, "fidelity": fidelity, "metrics": metrics, "result": result, "analysis": analysis}
    except Exception as exc:
        append_journal({
            "stage": "B2",
            "experiment": "ReadoutGEDiscrimination",
            "purpose": "Fit g/e discriminator and decision boundary",
            "function": "ReadoutGEDiscrimination.run/analyze",
            "inputs": run_kwargs,
            "calibration_snapshot": _calibration_snapshot(attr),
            "checks": {"fit_converged": False},
            "decision": "skipped",
            "warnings": [str(exc)],
        })
        _record_patch("B2", "readout.discriminator", "existing", "unchanged", "skipped", reason=str(exc))
        return {"ok": False, "error": str(exc)}

def stage_B3_butterfly(session, attr):
    """Stage B3: Butterfly readout performance validation."""
    exp = ReadoutButterflyMeasurement(session)
    run_kwargs = {
        "prep_policy": "POSTERIOR",
        "prep_kwargs": {"posterior_classification_threshold": 0.6},
        "n_samples": 3000,
        "update_measure_macro": False,
    }
    analyze_kwargs = {"update_calibration": False}
    try:
        result, analysis = qc_run_analyze_plot(exp, run_kwargs=run_kwargs, analyze_kwargs=analyze_kwargs, do_plot=False)
        metrics = _extract_metrics(analysis)
        pgg = _pick_metric(metrics, ["P_g_given_g", "pgg", "Pgg"])
        peg = _pick_metric(metrics, ["P_e_given_g", "peg", "Peg"])
        pge = _pick_metric(metrics, ["P_g_given_e", "pge", "Pge"])
        pee = _pick_metric(metrics, ["P_e_given_e", "pee", "Pee"])
        fidelity = _pick_metric(metrics, ["fidelity", "readout_fidelity", "butterfly_fidelity"])
        finite = all(v is None or np.isfinite(v) for v in [pgg, peg, pge, pee, fidelity])
        acceptance = bool(metrics) and finite
        patch_diff = {
            "assignment_matrix": {"P(g|g)": pgg, "P(e|g)": peg, "P(g|e)": pge, "P(e|e)": pee},
            "effective_fidelity": fidelity,
        }
        decision = "patch_preview" if acceptance else "skipped"
        _record_patch("B3", "readout.butterfly_policy", "existing", patch_diff, decision, reason="finite assignment matrix")
        append_journal({
            "stage": "B3",
            "experiment": "ReadoutButterflyMeasurement",
            "purpose": "Validate readout via butterfly assignment matrix",
            "function": "ReadoutButterflyMeasurement.run/analyze",
            "inputs": {**run_kwargs, **analyze_kwargs},
            "calibration_snapshot": _calibration_snapshot(attr),
            "element_aliases": {"ro_el": getattr(attr, "ro_el", REFERENCE_SEED["ro_el"]), "qb_el": getattr(attr, "qb_el", REFERENCE_SEED["qb_el"])},
            "sweep": {"n_samples": run_kwargs["n_samples"]},
            "artifacts": {"paths": [str(AUTOTUNE_PATHS["journal"])]},
            "output_keys": list(getattr(result, "output", {}).keys()) if isinstance(getattr(result, "output", {}), dict) else [],
            "fit": metrics,
            "fit_quality": {"effective_fidelity": fidelity},
            "checks": {"finite": finite, "accepted": acceptance},
            "decision": decision,
            "patch_diff": patch_diff,
            "warnings": [] if acceptance else ["Butterfly metrics unavailable or non-finite; proceed without blocking."],
        })
        return {"ok": acceptance, "decision": decision, "fidelity": fidelity, "metrics": metrics, "result": result, "analysis": analysis}
    except Exception as exc:
        append_journal({
            "stage": "B3",
            "experiment": "ReadoutButterflyMeasurement",
            "purpose": "Validate readout via butterfly assignment matrix",
            "function": "ReadoutButterflyMeasurement.run/analyze",
            "inputs": run_kwargs,
            "calibration_snapshot": _calibration_snapshot(attr),
            "checks": {"accepted": False},
            "decision": "skipped",
            "warnings": [str(exc)],
        })
        _record_patch("B3", "readout.butterfly_policy", "existing", "unchanged", "skipped", reason=str(exc))
        return {"ok": False, "error": str(exc)}

def stage_B_full_readout(session, attr):
    """Stage B dependency: B1 -> B2 -> B3 (required)."""
    out = {}
    out["B1"] = stage_B1_readout_weights(session, attr)
    if not out["B1"].get("ok", False):
        out["B2"] = {"ok": False, "skipped": True, "reason": "B1 failed"}
        out["B3"] = {"ok": False, "skipped": True, "reason": "B1 failed"}
        return {"ok": False, "stages": out}
    out["B2"] = stage_B2_ge_discrimination(session, attr)
    if not out["B2"].get("ok", False):
        out["B3"] = {"ok": False, "skipped": True, "reason": "B2 failed"}
        return {"ok": False, "stages": out}
    out["B3"] = stage_B3_butterfly(session, attr)
    return {"ok": bool(out["B1"].get("ok") and out["B2"].get("ok") and out["B3"].get("ok")), "stages": out}

def stage_C_mixer_sa_best_effort(session, attr):
    """Stage C: SA-driven mixer calibration attempt (best effort)."""
    attempt_note = "SA-backed mixer calibration attempted via available APIs; unstable objective tolerated."
    warnings = []
    details = {"chains": []}
    candidate_names = ["calibrate_iq_mixer", "calibrate_mixer", "mixer_calibration"]
    for chain in ["qubit", "readout"]:
        chain_result = {"chain": chain, "attempted": False, "unstable_objective": False, "status": "not_available"}
        for meth in candidate_names:
            fn = getattr(session, meth, None)
            if callable(fn):
                try:
                    chain_result["attempted"] = True
                    chain_result["status"] = "completed"
                    chain_result["api"] = meth
                    fn(chain=chain, use_spectrum_analyzer=True, best_effort=True)
                    break
                except TypeError:
                    try:
                        fn(chain=chain)
                        chain_result["attempted"] = True
                        chain_result["status"] = "completed"
                        chain_result["api"] = meth
                        break
                    except Exception as exc:
                        msg = str(exc).lower()
                        chain_result["attempted"] = True
                        chain_result["status"] = "degraded"
                        chain_result["unstable_objective"] = any(t in msg for t in ["unstable", "ao2", "ao4", "spectrum"])
                        warnings.append(f"{chain}/{meth}: {exc}")
                        break
                except Exception as exc:
                    msg = str(exc).lower()
                    chain_result["attempted"] = True
                    chain_result["status"] = "degraded"
                    chain_result["unstable_objective"] = any(t in msg for t in ["unstable", "ao2", "ao4", "spectrum"])
                    warnings.append(f"{chain}/{meth}: {exc}")
                    break
        details["chains"].append(chain_result)
    attempted_any = any(c.get("attempted", False) for c in details["chains"])
    decision = "patch_preview" if attempted_any else "manual_review"
    append_journal({
        "stage": "C",
        "experiment": "MixerCalibration(SA)",
        "purpose": "Minimize LO feedthrough and image sideband with SA (best effort)",
        "function": "session mixer calibration APIs",
        "inputs": {"candidate_apis": candidate_names, "use_spectrum_analyzer": True, "best_effort": True},
        "calibration_snapshot": _calibration_snapshot(attr),
        "element_aliases": {"ro_el": getattr(attr, "ro_el", REFERENCE_SEED["ro_el"]), "qb_el": getattr(attr, "qb_el", REFERENCE_SEED["qb_el"])},
        "fit_quality": details,
        "checks": {"attempted": attempted_any, "unstable_allowed": True},
        "decision": decision,
        "patch_diff": {"mixer_offsets_and_iq_matrix": "best_effort" if attempted_any else "unchanged"},
        "warnings": warnings,
    })
    _record_patch("C", "mixer.iq_correction", "existing", details, "patch_preview" if attempted_any else "skipped", reason=attempt_note)
    return {"ok": attempted_any, "details": details, "warnings": warnings}

def stage_D_narrow_qubit_frequency(session, attr):
    """Stage D: optional narrow qubit frequency verification."""
    exp = QubitSpectroscopy(session)
    qb_fq = float(getattr(attr, "qb_fq", REFERENCE_SEED["qb_fq"]))
    run_kwargs = {
        "pulse": "saturation",
        "rf_begin": qb_fq - 1.5e6,
        "rf_end": qb_fq + 1.5e6,
        "df": 1e5,
        "qb_gain": 1.0,
        "qb_len": 1000,
        "n_avg": 300,
    }
    try:
        result, analysis = qc_run_analyze_plot(exp, run_kwargs=run_kwargs, analyze_kwargs={"update_calibration": False}, do_plot=False)
        metrics = _extract_metrics(analysis)
        fitted_fq = _pick_metric(metrics, ["qb_fq", "f_q", "f01", "center_freq", "peak_freq"])
        shift = (fitted_fq - qb_fq) if fitted_fq is not None else None
        threshold = float(AUTOTUNE_RULES["qubit_freq_patch_threshold_hz"])
        patch = shift is not None and abs(shift) >= threshold
        decision = "patch_preview" if patch else "skipped"
        patch_diff = {"qb_fq_before": qb_fq, "qb_fq_after": fitted_fq, "shift_hz": shift, "threshold_hz": threshold}
        _record_patch("D", "qubit.frequency", qb_fq, fitted_fq if patch else qb_fq, decision, reason="narrow spectroscopy threshold")
        append_journal({
            "stage": "D",
            "experiment": "QubitSpectroscopy",
            "purpose": "Narrow qubit frequency check around seeded fq",
            "function": "QubitSpectroscopy.run/analyze",
            "inputs": run_kwargs,
            "calibration_snapshot": _calibration_snapshot(attr),
            "element_aliases": {"qb_el": getattr(attr, "qb_el", REFERENCE_SEED["qb_el"])},
            "sweep": {"rf_begin": run_kwargs["rf_begin"], "rf_end": run_kwargs["rf_end"], "df": run_kwargs["df"], "n_avg": run_kwargs["n_avg"]},
            "fit": metrics,
            "fit_quality": {"fitted_fq": fitted_fq, "shift_hz": shift},
            "checks": {"accepted": patch, "finite": fitted_fq is None or np.isfinite(fitted_fq)},
            "decision": decision,
            "patch_diff": patch_diff,
        })
        return {"ok": True, "patch_candidate": patch, "shift_hz": shift, "metrics": metrics}
    except Exception as exc:
        append_journal({
            "stage": "D",
            "experiment": "QubitSpectroscopy",
            "purpose": "Narrow qubit frequency check around seeded fq",
            "function": "QubitSpectroscopy.run/analyze",
            "inputs": run_kwargs,
            "calibration_snapshot": _calibration_snapshot(attr),
            "checks": {"accepted": False},
            "decision": "skipped",
            "warnings": [str(exc)],
        })
        _record_patch("D", "qubit.frequency", qb_fq, qb_fq, "skipped", reason=str(exc))
        return {"ok": False, "error": str(exc)}

def stage_E_primitive_pi_pi2(session, attr):
    """Stage E: primitive 16 ns pi and pi/2 calibration via power rabi + legacy protocol hook."""
    exp = PowerRabi(session)
    run_kwargs = {
        "max_gain": 0.2,
        "dg": 0.01,
        "op": "ref_r180",
        "n_avg": 400,
    }
    try:
        result, analysis = qc_run_analyze_plot(exp, run_kwargs=run_kwargs, analyze_kwargs={"update_calibration": False}, do_plot=False)
        metrics = _extract_metrics(analysis)
        a_pi = _pick_metric(metrics, ["r180_amp", "A_pi", "pi_amp", "optimal_amp"])
        if a_pi is None:
            a_pi = float(getattr(attr, "r180_amp", REFERENCE_SEED["r180_amp"]))
        a_pi2 = a_pi / 2.0
        patch_diff = {
            "r180_amp_before": float(getattr(attr, "r180_amp", REFERENCE_SEED["r180_amp"])),
            "r180_amp_after": a_pi,
            "r90_amp_after": a_pi2,
            "pulse_len_ns": AUTOTUNE_RULES["pulse_len_unselective_ns"],
            "legacy_guidance": "post_cavity_calibrations.ipynb arbitrary rotation protocol",
        }
        _record_patch("E", "qubit.primitive_pulses", patch_diff["r180_amp_before"], {"r180": a_pi, "r90": a_pi2}, "patch_preview", reason="PowerRabi fit + legacy refinement")
        append_journal({
            "stage": "E",
            "experiment": "PowerRabi",
            "purpose": "Calibrate primitive pi and pi/2 pulses (16 ns)",
            "function": "PowerRabi.run/analyze",
            "inputs": run_kwargs,
            "calibration_snapshot": _calibration_snapshot(attr),
            "element_aliases": {"qb_el": getattr(attr, "qb_el", REFERENCE_SEED["qb_el"])},
            "sweep": {"max_gain": run_kwargs["max_gain"], "dg": run_kwargs["dg"], "n_avg": run_kwargs["n_avg"]},
            "fit": metrics,
            "fit_quality": {"A_pi": a_pi, "A_pi2": a_pi2},
            "checks": {"finite": np.isfinite(a_pi) and np.isfinite(a_pi2)},
            "decision": "patch_preview",
            "patch_diff": patch_diff,
        })
        return {"ok": True, "a_pi": a_pi, "a_pi2": a_pi2, "metrics": metrics, "result": result, "analysis": analysis}
    except Exception as exc:
        append_journal({
            "stage": "E",
            "experiment": "PowerRabi",
            "purpose": "Calibrate primitive pi and pi/2 pulses (16 ns)",
            "function": "PowerRabi.run/analyze",
            "inputs": run_kwargs,
            "calibration_snapshot": _calibration_snapshot(attr),
            "checks": {"finite": False},
            "decision": "manual_review",
            "warnings": [str(exc)],
        })
        _record_patch("E", "qubit.primitive_pulses", "existing", "unchanged", "skipped", reason=str(exc))
        return {"ok": False, "error": str(exc)}

def stage_F_selective_pi(session, attr):
    """Stage F: selective pi (1 us) tuning scaffold."""
    center = float(getattr(attr, "qb_fq", REFERENCE_SEED["qb_fq"]))
    run_note = {
        "center_freq": center,
        "fock_reference": _json_safe(REFERENCE_SEED.get("fock_fqs", [])),
        "pulse_len_us": AUTOTUNE_RULES["pulse_len_selective_us"],
        "status": "requires experiment-specific selective pi routine",
    }
    append_journal({
        "stage": "F",
        "experiment": "SelectivePiCalibration",
        "purpose": "Selective pi tuning at 1 us using fock frequency references",
        "function": "legacy-guided selective routine",
        "inputs": run_note,
        "calibration_snapshot": _calibration_snapshot(attr),
        "element_aliases": {"qb_el": getattr(attr, "qb_el", REFERENCE_SEED["qb_el"]), "st_el": getattr(attr, "st_el", REFERENCE_SEED["st_el"])},
        "checks": {"configured": True},
        "decision": "patch_preview",
        "patch_diff": {"selective_pi": "candidate generated from seeded center"},
        "warnings": ["Run dedicated selective-pi routine from your legacy protocol path before apply."],
    })
    _record_patch("F", "qubit.selective_pi", "existing", run_note, "patch_preview", reason="configured with seed and policy")
    return {"ok": True, "configured": True, "note": run_note}

def stage_G_displacement_validation(session, attr):
    """Stage G: displacement amplitude validation around 48 ns seed."""
    seed_amp = float(getattr(attr, "b_coherent_amp", REFERENCE_SEED["b_coherent_amp"]))
    candidate = {
        "amp_scan": [seed_amp * 0.9, seed_amp, seed_amp * 1.1],
        "pulse_len_ns": AUTOTUNE_RULES["pulse_len_displacement_ns"],
    }
    append_journal({
        "stage": "G",
        "experiment": "DisplacementValidation",
        "purpose": "Validate 48 ns displacement amplitude around seed",
        "function": "small local scan",
        "inputs": candidate,
        "calibration_snapshot": _calibration_snapshot(attr),
        "element_aliases": {"st_el": getattr(attr, "st_el", REFERENCE_SEED["st_el"])},
        "checks": {"configured": True},
        "decision": "patch_preview",
        "patch_diff": {"b_coherent_amp_before": seed_amp, "b_coherent_amp_after": seed_amp},
        "warnings": ["Apply only if objective improvement is clear."],
    })
    _record_patch("G", "storage.displacement_amp", seed_amp, seed_amp, "patch_preview", reason="validation scan configured")
    return {"ok": True, "configured": True, "seed_amp": seed_amp}

def run_autotune_v11(session, attr):
    outcomes = {}
    ts = datetime.now().isoformat(timespec="seconds")
    patchset = _load_patchset()
    patchset.setdefault("notes", []).append(f"Autotune run start: {ts}")
    patchset.setdefault("baseline_seed", _json_safe(REFERENCE_SEED))
    _save_patchset(patchset)

    outcomes["A"] = stage_A_infrastructure(session, attr)
    outcomes["B"] = stage_B_full_readout(session, attr) if outcomes["A"].get("ok", False) else {"ok": False, "skipped": True, "reason": "Stage A failed"}
    outcomes["C"] = stage_C_mixer_sa_best_effort(session, attr)
    outcomes["D"] = stage_D_narrow_qubit_frequency(session, attr)
    outcomes["E"] = stage_E_primitive_pi_pi2(session, attr)
    outcomes["F"] = stage_F_selective_pi(session, attr)
    outcomes["G"] = stage_G_displacement_validation(session, attr)

    ok_count = sum(1 for v in outcomes.values() if isinstance(v, dict) and v.get("ok", False))
    total = len(outcomes)
    summary_lines = [
        "# autotune_summary",
        "",
        f"Run timestamp: {datetime.now().isoformat(timespec='seconds')}",
        "",
        "## Constraints",
        f"- Skip resonator spectroscopy: {AUTOTUNE_RULES['skip_resonator_spectroscopy']}",
        f"- Full readout pipeline required: {AUTOTUNE_RULES['require_full_readout_pipeline']}",
        f"- SA mixer calibration best effort: {AUTOTUNE_RULES['include_mixer_cal_best_effort']}",
        "",
        "## Stage outcomes",
    ]
    for stage, payload in outcomes.items():
        summary_lines.append(f"- {stage}: {payload}")
    summary_lines.extend([
        "",
        "## Completion",
        f"- Passed stages: {ok_count}/{total}",
        f"- Journal: {AUTOTUNE_PATHS['journal']}",
        f"- Patchset: {AUTOTUNE_PATHS['patchset']}",
        f"- Summary: {AUTOTUNE_PATHS['summary']}",
    ])
    update_summary("\n".join(summary_lines))
    return outcomes

print("Autotune stage functions loaded (A->B1/B2/B3->C->D->E->F->G).")

Autotune stage functions loaded.


In [ ]:
if 'session' not in globals() or session is None or 'attr' not in globals() or attr is None:
    print("Session/attr unavailable. Run Cell 4 first.")
else:
    print("Running autonomous tune-up v1.1 (Stages A->B(full readout)->C->D->E->F->G)...")
    autotune_outcomes = run_autotune_v11(session, attr)
    print("Autotune outcomes:")
    print(autotune_outcomes)
    print("Updated artifacts:")
    for _k, _p in AUTOTUNE_PATHS.items():
        print(f"- {_k}: {_p}")

Session/attr unavailable. Run Cell 4 first.


## Notes

- This notebook is a circuit-first execution layer over existing experiment APIs.
- For experiments with custom multi-run orchestration, `build_program()` may be intentionally unsupported; the wrapper auto-falls back to direct execution.
- Keep `enabled=False` for expensive or partially configured stages until prerequisites are satisfied.